# MongoDB connection test

Run these cells from top to bottom. The `ping` cell verifies the Atlas connection before `insert_many`, so connection problems fail quickly with a useful message.

In [ ]:
import os
from getpass import getpass

import certifi
import pandas as pd
import pymongo
from pymongo.errors import PyMongoError, ServerSelectionTimeoutError

In [ ]:
from pathlib import Path

DB_NAME = "US_VISA"
COLLECTION_NAME = "visa_data"

CSV_PATH = Path("EasyVisa.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("notebooks") / "EasyVisa.csv"

Set `MONGODB_URL` in your environment, or paste it when prompted. Keep the URI private.

In [ ]:
import ssl

print("Python SSL:", ssl.OPENSSL_VERSION)
print("PyMongo:", pymongo.version)
print("certifi CA file:", certifi.where())

connection_url = os.getenv("MONGODB_URL")

if not connection_url:
    connection_url = getpass("Paste your MongoDB connection string: ")

if not connection_url.startswith(("mongodb://", "mongodb+srv://")):
    raise ValueError("MONGODB_URL must start with mongodb:// or mongodb+srv://")

In [ ]:
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=5000,
    connectTimeoutMS=20000,
    socketTimeoutMS=20000,
    serverSelectionTimeoutMS =20000,
)

try:
    client.admin.command("ping")
    print("MongoDB connection successful")
except ServerSelectionTimeoutError as exc:
    message = str(exc)
    if "SSL handshake failed" in message or "tlsv1 alert internal error" in message:
        raise RuntimeError(
            "MongoDB Atlas rejected the TLS handshake. Fix these first: "
            "1) Atlas Network Access must include your current public IP, "
            "2) your cluster must be running, "
            "3) your URI must be the Driver connection string for Python, and "
            "4) PyMongo/dnspython/certifi should be updated in this notebook kernel. "
            "For a quick test only, add 0.0.0.0/0 in Atlas Network Access, then rerun this cell."
        ) from exc
    raise RuntimeError(
        "Could not connect to a writable MongoDB server. In MongoDB Atlas, make sure "
        "your cluster is running, your current IP is added under Network Access, "
        "and your username/password in MONGODB_URL are correct."
    ) from exc

In [ ]:
df = pd.read_csv(CSV_PATH)
data = df.to_dict(orient="records")

print(f"Loaded {len(data)} rows from {CSV_PATH}")
df.head()

In [ ]:
database = client[DB_NAME]
collection = database[COLLECTION_NAME]

print(f"Using database={DB_NAME!r}, collection={COLLECTION_NAME!r}")

In [ ]:
if not data:
    raise ValueError("No records found to insert")

result = collection.insert_many(data)
print(f"Inserted {len(result.inserted_ids)} records")